<a href="https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

# 1. Ranked actions + reason codes

### Priority Action Queue Strategy
The model translates raw decay risk probabilities into structured action archetypes. Out of 30,000 evaluated URLs, **22,116 URLs (73.7%)** are flagged for active intervention, ranked by decay risk score and pre-period exposure (`impressions_90d`).

### Queue Archetype & Reason Code Breakdown
* **`EDITORIAL_REWRITE` ($n = 10,308$ / `REASON_GENERAL_ORGANIC_DECLINE`):** Primary driver of flagged content. URLs exhibit steady traffic degradation across general tiers, requiring core content updates and search intent realignment.
* **`TECHNICAL_KEYWORDS_AUDIT` ($n = 8,428$ / `REASON_POOR_RANKING_POSITION`):** URLs ranking past Page 2 (`avg_position` $> 15.0$). Action: Audit keyword targeting, canonical tags, and internal link architecture.
* **`MONITOR_STABLE` ($n = 7,884$ / `REASON_HEALTHY_TRAFFIC_BASELINE`):** Risk score $< 0.50$. Retain in baseline queue without active editorial intervention.
* **`CTR_SNIPPET_OPTIMIZATION` ($n = 3,319$ / `REASON_HIGH_IMPRESSIONS_LOW_CTR`):** Page 1 URLs (`avg_position` $\le 10.0$) with low CTR ($< 2\%$). Action: Rewrite title tags, meta descriptions, and featured snippet formatting.
* **`CONTENT_EXPANSION_REFRESH` ($n = 61$ / `REASON_STALE_CONTENT_DECAY`):** Content un-updated for $\ge 180$ days experiencing organic decline. Action: Expand subtopic depth and refresh outdated figures.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# Create necessary directories for exports
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Load raw dataset directly from repository
raw_url = "https://raw.githubusercontent.com/hamzakhattak-AIDev/flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)

# Formulate binary target and select historical feature set
df["target"] = df["trend_direction"].str.lower().eq("down").astype(int)
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "clicks_90d"
]

# Train Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(df[feature_cols], df["target"])

# Predict risk probabilities and class
df["decay_risk_score"] = rf_model.predict_proba(df[feature_cols])[:, 1]

# Assign Archetype and Reason Code Mapping
def assign_archetype(row):
    risk = row["decay_risk_score"]
    days = row["days_since_last_update"]
    pos = row["avg_position"]
    ctr = row["ctr"]

    if risk >= 0.50:
        if pos <= 10.0 and ctr < 0.02:
            return "CTR_SNIPPET_OPTIMIZATION", "REASON_HIGH_IMPRESSIONS_LOW_CTR"
        elif days >= 180:
            return "CONTENT_EXPANSION_REFRESH", "REASON_STALE_CONTENT_DECAY"
        elif pos > 15.0:
            return "TECHNICAL_KEYWORDS_AUDIT", "REASON_POOR_RANKING_POSITION"
        else:
            return "EDITORIAL_REWRITE", "REASON_GENERAL_ORGANIC_DECLINE"
    else:
        return "MONITOR_STABLE", "REASON_HEALTHY_TRAFFIC_BASELINE"

results = df.apply(assign_archetype, axis=1)
df["recommended_action"] = [r[0] for r in results]
df["reason_code"] = [r[1] for r in results]

# Sort priority queue by highest risk score and traffic volume
queue_df = df.sort_values(by=["decay_risk_score", "impressions_90d"], ascending=[False, False])

print("=== QUEUE ARCHETYPE DISTRIBUTION ===")
print(queue_df["recommended_action"].value_counts())
print("\n=== REASON CODE DISTRIBUTION ===")
print(queue_df["reason_code"].value_counts())

=== QUEUE ARCHETYPE DISTRIBUTION ===
recommended_action
EDITORIAL_REWRITE            10308
TECHNICAL_KEYWORDS_AUDIT      8428
MONITOR_STABLE                7884
CTR_SNIPPET_OPTIMIZATION      3319
CONTENT_EXPANSION_REFRESH       61
Name: count, dtype: int64

=== REASON CODE DISTRIBUTION ===
reason_code
REASON_GENERAL_ORGANIC_DECLINE     10308
REASON_POOR_RANKING_POSITION        8428
REASON_HEALTHY_TRAFFIC_BASELINE     7884
REASON_HIGH_IMPRESSIONS_LOW_CTR     3319
REASON_STALE_CONTENT_DECAY            61
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

# 2. Intended use and limits

### Intended Use
This playbook functions strictly as a **non-production decision-support tool** for SEO managers and content editorial teams. It automates top-of-funnel triage across multi-client URL inventories.

### Triage Capacity Tiers & Operational Limits
* **High Priority ($n = 461$ URLs, Risk $\ge 0.70$):** Immediate editorial review queue representing urgent organic loss.
* **Medium Priority ($n = 21,655$ URLs, $0.50 \le \text{Risk} < 0.70$):** Secondary queue for scheduled content refreshes.
* **Low Priority / Monitor ($n = 7,884$ URLs, Risk $< 0.50$):** Healthy baseline content.
* **Operational Limits:** The playbook does not auto-generate or publish copy. Furthermore, evaluation performance degrades when applied to newly onboarded domains with atypical impression distributions (observed domain variation: mean client risk spans from 0.0057 to 0.6737).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check score distribution across risk thresholds for operational triage
high_risk = queue_df[queue_df["decay_risk_score"] >= 0.70]
mod_risk = queue_df[(queue_df["decay_risk_score"] >= 0.50) & (queue_df["decay_risk_score"] < 0.70)]
low_risk = queue_df[queue_df["decay_risk_score"] < 0.50]

print("=== TRIAGE TIER BREAKDOWN ===")
print(f"High Priority (Risk >= 0.70): {len(high_risk)} URLs")
print(f"Medium Priority (0.50 <= Risk < 0.70): {len(mod_risk)} URLs")
print(f"Low Priority / Monitor (Risk < 0.50): {len(low_risk)} URLs")

=== TRIAGE TIER BREAKDOWN ===
High Priority (Risk >= 0.70): 461 URLs
Medium Priority (0.50 <= Risk < 0.70): 21655 URLs
Low Priority / Monitor (Risk < 0.50): 7884 URLs


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

# 3. Human review + the no-go list

### Human-in-the-Loop Review Rules
* **Page 1 Top 5 Triage ($n = 2,255$ URLs):** A total of 2,255 URLs rank in top 5 positions yet carry high decay risk scores ($\ge 0.50$). These high-exposure assets require mandatory manual review by a senior strategist before executing any copy edits to avoid destabilizing existing conversion funnels.
* **Intent Drift Check:** Editors must manually verify that declining traffic reflects fixable content staleness rather than structural search intent shifts.

### The No-Go List (Never Automate)
1. **High-Converting Product/Landing Pages:** Core conversion pages must never undergo automated rewrites.
2. **YMYL & Legal Content:** Medical, financial, and legal articles require domain expert validation.
3. **Canonical and Domain Migrations:** URL redirects and site architecture changes must remain manually governed.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Identify URLs in high position but flagged for action to simulate human review trigger
human_review_queue = queue_df[(queue_df["decay_risk_score"] >= 0.50) & (queue_df["avg_position"] <= 5.0)]

print(f"Total Flagged URLs on Page 1 Top 5 (Mandatory Human Review): {len(human_review_queue)}")
print("Sample High-Value Page 1 URLs Needing Manual Review:")
print(human_review_queue[["client_id", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "decay_risk_score"]].head(3).to_string(index=False))

Total Flagged URLs on Page 1 Top 5 (Mandatory Human Review): 2255
Sample High-Value Page 1 URLs Needing Manual Review:
        client_id  days_since_last_update  impressions_90d  avg_position  ctr  decay_risk_score
client_8527a891e2                     104               98           0.5 0.00          0.777910
client_4e07408562                     104             3182           0.9 0.06          0.769079
client_19581e27de                     104             2695           0.2 0.07          0.766183


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

# 4. Monitoring / retrain triggers

### Drift Monitoring Baseline & Triggers
* **Client Portfolio Risk Variance:** Baseline population risk averages **0.5419**, but varies from **0.0057** to **0.6737** across client portfolios.
* **Core Algorithm Update Trigger:** Retrain model weights following major search engine algorithm updates to account for systemic baseline position shifts.
* **Quarterly Schedule:** Retrain the Random Forest model every 90 days.
* **Error Rate Threshold:** If human review sampling reveals a false positive rate exceeding 35% over two consecutive audit weeks, halt queue exports and re-evaluate feature definitions.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Calculate client-level variance to define drift thresholds
client_risk_variance = queue_df.groupby("client_id")["decay_risk_score"].mean()

print("=== CLIENT RISK SCORE VARIANCE (DRIFT MONITORING BASELINE) ===")
print(f"Max Client Avg Decay Risk: {client_risk_variance.max():.4f}")
print(f"Min Client Avg Decay Risk: {client_risk_variance.min():.4f}")
print(f"Overall Population Mean Risk: {queue_df['decay_risk_score'].mean():.4f}")

=== CLIENT RISK SCORE VARIANCE (DRIFT MONITORING BASELINE) ===
Max Client Avg Decay Risk: 0.6737
Min Client Avg Decay Risk: 0.0057
Overall Population Mean Risk: 0.5419


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

# 5. Exports for the paper

### Deliverables Generated for Capstone Paper
* **Ranked Queue Export:** `work/outputs/ranked_action_queue.csv` (30,000 rows exported; un-tracked by Git per repo leak-guard rules).
* **Figure Deliverable:** `work/figures/playbook_action_archetypes.png` (Visualization of archetype allocations).
* **Metrics Receipt:** `work/figures/playbook_metrics.json` (Structured JSON containing total evaluation counts and archetype distributions).

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import matplotlib.pyplot as plt
import json

# 1. Export figure to work/figures/
plt.figure(figsize=(8, 4))
queue_df["recommended_action"].value_counts().plot(kind="barh", color="#2b5c8f")
plt.title("Content Action Playbook: Archetype Breakdown", fontsize=12, fontweight="bold")
plt.xlabel("URL Count")
plt.tight_layout()

fig_path = "work/figures/playbook_action_archetypes.png"
plt.savefig(fig_path, dpi=300)
plt.close()
print(f"Figure exported: {fig_path}")

# 2. Export ranked queue to work/outputs/ (ignored by git leak-guard)
if "url_hash" not in queue_df.columns:
    queue_df["url_hash"] = [f"url_{i:05d}" for i in range(len(queue_df))]

export_cols = [
    "url_hash", "client_id", "days_since_last_update",
    "impressions_90d", "avg_position", "ctr",
    "decay_risk_score", "recommended_action", "reason_code"
]

csv_path = "work/outputs/ranked_action_queue.csv"
queue_df[export_cols].to_csv(csv_path, index=False)
print(f"Ranked queue exported: {csv_path} ({len(queue_df)} rows)")

# 3. Export metrics JSON receipt
metrics = {
    "total_urls_evaluated": int(len(queue_df)),
    "flagged_for_action": int((queue_df["decay_risk_score"] >= 0.50).sum()),
    "archetype_counts": queue_df["recommended_action"].value_counts().to_dict(),
    "reason_code_counts": queue_df["reason_code"].value_counts().to_dict()
}

json_path = "work/figures/playbook_metrics.json"
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=4)
print(f"Metrics JSON receipt exported: {json_path}")

Figure exported: work/figures/playbook_action_archetypes.png
Ranked queue exported: work/outputs/ranked_action_queue.csv (30000 rows)
Metrics JSON receipt exported: work/figures/playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.